# 🍃 Aula 06 — Modelo de Documentos: JSON/BSON e Modelagem no MongoDB (*Embedding vs. Referencing*)
**Curso:** Tecnologia em Análise e Desenvolvimento de Sistemas (ADS) — IFCE Campus Tauá  
**Disciplina:** Banco de Dados Não-Relacionais (ADS16)  
**Docente:** Prof. Me. Reginaldo Fernandes  

---

## 🎯 Objetivos deste Notebook Prático
1. Inspecionar na prática o formato **BSON (*Binary JSON*)**, a estrutura de **12 bytes** do `ObjectId` e a precisão financeira do `Decimal128` contra erros de ponto flutuante (`float` IEEE 754).
2. Modelar o catálogo polimórfico do e-commerce **ShopGrids** (Notebooks, Camisetas e Livros na mesma coleção `produtos`) usando **Embedding 1:1**.
3. Implementar o **Subset Pattern** + **Referencing 1:Many** para avaliações de produtos, evitando *Unbounded Arrays* (estouro de 16 MB).
4. Implementar a coleção `pedidos` combinando **Extended Reference** e **Snapshot Pattern**, provando por que pedidos fiscais nunca devem referenciar preços mutáveis sem congelamento.

In [ ]:
# 1. Instalação de pacotes (necessário apenas se estiver rodando no Google Colab)
!pip install -q pymongo mongomock

In [ ]:
# 2. Raio-X do BSON: Float IEEE 754 vs. Decimal128 e Anatomia dos 12 Bytes do ObjectId
import json
from datetime import datetime, timezone
from decimal import Decimal
from bson import BSON, ObjectId
from bson.decimal128 import Decimal128

print('=== 1. POR QUE NÃO USAR FLOAT COMUM PARA PREÇOS E NOTAS FISCAIS? ===')
print(f'Float binário (0.1 + 0.2)   : {0.1 + 0.2:.17f}  <-- Erro de arredondamento!')
print(f'BSON Decimal128 (0.1 + 0.2) : {Decimal128(Decimal("0.1") + Decimal("0.2"))}                <-- Precisão exata!\n')

print('=== 2. ANATOMIA DOS 12 BYTES DO OBJECTID ===')
oid = ObjectId()
b = oid.binary
print(f'ObjectId (24 hex chars)     : {oid}')
print(f'  [Bytes 0..3] Timestamp    : {b[0:4].hex()} -> Criado em: {oid.generation_time.isoformat()}')
print(f'  [Bytes 4..8] Aleatório    : {b[4:9].hex()} (Máquina/Processo)')
print(f'  [Bytes 9..11] Contador    : {b[9:12].hex()} (Incremento atômico)\n')

print('=== 3. SERIALIZAÇÃO EM DISCO: JSON TEXTUAL vs. BSON BINÁRIO ===')
doc = {'_id': oid, 'sku': 'NOTE-ACER-15', 'preco': Decimal128('4599.90'), 'estoque': 15}
bson_bytes = BSON.encode(doc)
print(f'Tamanho em BSON Binário     : {len(bson_bytes)} bytes (com cabeçalhos de tamanho para salto rápido)')

In [ ]:
# 3. Conectando ao MongoDB (Local na porta 27017 ou Fallback em Memória para Google Colab)
from pymongo import MongoClient
import mongomock

try:
    client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=2000)
    client.admin.command('ping')
    print('✅ Conectado ao servidor MongoDB real em localhost:27017!')
except Exception:
    client = mongomock.MongoClient()
    print('ℹ️ Servidor MongoDB local não detectado (ex: Google Colab). Usando motor mongomock em memória!')

db = client['ecommerce_shopgrids']
db.produtos.drop()
db.avaliacoes.drop()
db.clientes.drop()
db.pedidos.drop()

In [ ]:
# 4. Polimorfismo + Embedding (1:1) + Subset Pattern na Coleção 'produtos'
id_notebook = ObjectId()
id_camiseta = ObjectId()

db.produtos.insert_many([
    {
        '_id': id_notebook,
        'sku': 'NOTE-ACER-15',
        'nome': 'Notebook Gamer Acer Nitro V15',
        'categoria': 'Eletronicos',
        'preco': Decimal128('4599.90'),
        'estoque': 15,
        # EMBEDDING 1:1 (Polimorfismo: especificações técnicas de hardware)
        'especificacoes': {
            'processador': 'Intel Core i7-13620H',
            'ram_gb': 16,
            'gpu': 'RTX 4050 6GB'
        },
        # SUBSET PATTERN: guardamos apenas o resumo e as Top 2 avaliações recentes no produto!
        'resumo_avaliacoes': {'media': 4.9, 'total': 2},
        'ultimas_avaliacoes': [
            {'autor': 'Lucas', 'nota': 5, 'comentario': 'Excelente para Docker e IA!'}
        ]
    },
    {
        '_id': id_camiseta,
        'sku': 'CAM-NOSQL-IFCE',
        'nome': 'Camiseta Dev Polyglot Persistence IFCE',
        'categoria': 'Vestuario',
        'preco': Decimal128('69.90'),
        'estoque': 80,
        # EMBEDDING 1:1 (Polimorfismo: atributos de vestuário na mesma coleção!)
        'especificacoes': {
            'tecido': '100% Algodão Egípcio',
            'cor': 'Preta',
            'tamanhos': ['P', 'M', 'G', 'GG']
        },
        'resumo_avaliacoes': {'media': 5.0, 'total': 1},
        'ultimas_avaliacoes': []
    }
])

for p in db.produtos.find():
    print(f"Produto: {p['sku']} | Preço: R$ {p['preco']} | Especificações: {p['especificacoes']}")

In [ ]:
# 5. Pedidos com Extended Reference + Snapshot Pattern (Proteção Fiscal)
id_cliente = ObjectId()
db.clientes.insert_one({
    '_id': id_cliente,
    'nome': 'Ana Beatriz Costa',
    'cpf': '123.456.789-00',
    'enderecos': [{'apelido': 'Casa', 'rua': 'Av. Chermont Alves, 100', 'cidade': 'Tauá-CE'}]
})

# Criamos o pedido congelando o preço do produto no momento da compra (Snapshot):
prod_comprado = db.produtos.find_one({'_id': id_notebook})
db.pedidos.insert_one({
    'codigo_pedido': 'PED-2026-0001',
    'cliente': {'cliente_id': id_cliente, 'nome': 'Ana Beatriz Costa'},
    'itens': [{
        'produto_id': prod_comprado['_id'],
        'sku': prod_comprado['sku'],
        'nome': prod_comprado['nome'],
        'preco_unitario_na_compra': prod_comprado['preco'],
        'quantidade': 1
    }],
    'status': 'PAGO'
})

# Agora simulamos um aumento de preço na vitrine (de R$ 4.599,90 para R$ 5.299,90):
db.produtos.update_one({'_id': id_notebook}, {'$set': {'preco': Decimal128('5299.90')}})

preco_vitrine_hoje = db.produtos.find_one({'_id': id_notebook})['preco']
preco_nota_fiscal = db.pedidos.find_one({'codigo_pedido': 'PED-2026-0001'})['itens'][0]['preco_unitario_na_compra']

print(f'Preço ATUAL na vitrine (db.produtos)            : R$ {preco_vitrine_hoje}')
print(f'Preço HISTÓRICO na Nota Fiscal (db.pedidos)     : R$ {preco_nota_fiscal}  <-- Protegido pelo Snapshot Pattern!')